Analyse de la Valeur Client (Segmentation RFM)
Objectif : Classer les clients en catégories (VIP, Réguliers, À risque, Perdus) selon leur comportement d'achat.

Récence (R) : Date de la dernière commande.

Fréquence (F) : Nombre total de commandes.

Montant (M) : Chiffre d'affaires total généré.

In [ ]:
import sys
import os
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration du chemin pour importer ta fonction de connexion
sys.path.append(os.path.abspath(os.path.join('..')))
from src.analysis.connection import get_db_data 

# 1. Extraction des données
query = "SELECT customerNumber, MAX(orderDate) as last_order_date, COUNT(orderNumber) as frequency, SUM(quantityOrdered * priceEach) as monetary FROM orders JOIN orderdetails USING (orderNumber) GROUP BY customerNumber"
df_rfm = get_db_data(query)
df_rfm['last_order_date'] = pd.to_datetime(df_rfm['last_order_date'])

# 2. Calculs de la Récence
ref_date = df_rfm['last_order_date'].max() + dt.timedelta(days=1)
df_rfm['recency'] = (ref_date - df_rfm['last_order_date']).dt.days

# 3. Scoring par Quartiles
df_rfm['R'] = pd.qcut(df_rfm['recency'], 4, labels=[4, 3, 2, 1])
df_rfm['F'] = pd.qcut(df_rfm['frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
df_rfm['M'] = pd.qcut(df_rfm['monetary'], 4, labels=[1, 2, 3, 4])

# 4. Fonction de Segmentation
def assign_segment(row):
    total = int(row['R']) + int(row['F']) + int(row['M'])
    if total >= 10: return 'VIP'
    elif total >= 7: return 'Régulier'
    elif total >= 5: return 'À risque'
    else: return 'Perdu'

df_rfm['Segment'] = df_rfm.apply(assign_segment, axis=1)

# 5. Visualisation
plt.figure(figsize=(10, 6))
sns.countplot(data=df_rfm, x='Segment', palette='viridis', order=['VIP', 'Régulier', 'À risque', 'Perdu'])
plt.title('Distribution des Segments Clients')

# Sauvegarde de l'image
save_path = os.path.join('..', 'visualisations', 'rfm_segmentation.png')
plt.savefig(save_path, bbox_inches='tight', dpi=300)
plt.show()
